In [ ]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "Car details v3.csv"

# Load the latest version
data = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "nehalbirla/vehicle-dataset-from-cardekho",
  file_path,
 ) 

O vehicle-dataset do Kaggle possui 4 datasets em sua base, escolhemos o dataset "Car details v3" por se encaixar melhor na proposta do projeto e conter conteúdo suficiente para uma anáise exploratória consistente. Ao que se observa em uma visualização comparativa com os outros 3. 

Alguns critérios do enunciado voltados à classificação não se aplicam diretamente, como desbalanceamento de classes e separabilidade entre classes, nesses casos, analisaremos a distribuição do target e padrões associados ao preço

In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data.head()

In [ ]:
data.shape  #Para vermos numero de linhas e colunas
data.info 

O dataset selecionado contém 8128 linhas ou registros e 13 váriaveis ou colunas. A análise inicial permite que indentifiquemos o target "selling_price" e visualizar a estrutura geral das features dispíveis. Nas próximas etapas, vamos analisar cada variável quanto ao significado, tipo e qualidade dos dados.

TARGET -> selling_price

Dicionário das variáveis:

| Variável        | Significado                                                                   | Tipo conceitual                                              | Papel      |
| --------------- | ----------------------------------------------------------------------------- | ------------------------------------------------------------ | ---------- |
| `name`          | Nome/modelo do veículo, incluindo normalmente marca e versão                  | Categórica nominal                                           | Feature    |
| `year`          | Ano de fabricação/registro do veículo                                         | Numérica discreta/temporal                                   | Feature    |
| `selling_price` | Preço de venda registrado para o veículo                                      | Numérica                                                     | **Target** |
| `km_driven`     | Quilometragem já percorrida pelo veículo                                      | Numérica                                                     | Feature    |
| `fuel`          | Tipo de combustível utilizado pelo veículo                                    | Categórica nominal                                           | Feature    |
| `seller_type`   | Tipo de vendedor responsável pelo anúncio/venda                               | Categórica nominal                                           | Feature    |
| `transmission`  | Tipo de transmissão do veículo                                                | Categórica nominal                                           | Feature    |
| `owner`         | Indica a ordem do proprietário atual, como primeiro, segundo ou terceiro dono | Categórica ordinal                                           | Feature    |
| `mileage`       | Eficiência/consumo de combustível do veículo                                  | Numérica contínua conceitualmente                            | Feature    |
| `engine`        | Capacidade/cilindrada do motor, apresentada em `CC`                           | Numérica conceitualmente                                     | Feature    |
| `max_power`     | Potência máxima do motor, geralmente apresentada em `bhp`                     | Numérica contínua conceitualmente                            | Feature    |
| `torque`        | Torque do motor e, em muitos registros, a rotação em que ele é atingido       | Numérica conceitualmente, mas representada de forma composta | Feature    |
| `seats`         | Quantidade de assentos do veículo                                             | Numérica discreta                                            | Feature    |


A variável selling_price foi definida como target do projeto, caracterizando a futura tarefa como um problema de regressão. As outras variáveis serão inicialmente consideradas features candidatas. Algumas variáveis que representam grandezas numéricas aparecem armazenadas como texto, o data.info() retorna elas como object, aspecto que será investigado mais a frente durante a análise de qualidade e pré-processamento. Contudo, parecem ser referentes as ordens de grandeza.

In [23]:
data.isna().sum()

name               0
year               0
selling_price      0
km_driven          0
fuel               0
seller_type        0
transmission       0
owner              0
mileage          221
engine           221
max_power        215
torque           222
seats            221
dtype: int64

Valores ausentes em cinco variáveis: mileage, engine, max_power, torque e seats

In [24]:
data.duplicated().sum()

np.int64(1202)

1202 linhas idênticas

In [ ]:
data.nunique().sort_values() #Quantos valores diferentes existem em cada coluna.

transmission        2
seller_type         3
fuel                4
owner               5
seats               9
year               29
engine            121
max_power         322
mileage           393
torque            441
selling_price     677
km_driven         921
name             2058
dtype: int64

name tem 2058 valores diferentes, alta cardinalidade. É algo importante de se notar.

Vamos investigar esses aspectos após essa análise de qualidade dos dados, antes de definir estratégias de pré-processamento

Ainda não vamos usar drop duplicates pois em um dataset de carros duas linhas iguais podem significar duas coisas diferentes. Ou um mesmo anúncio repetido, ou dois carros diferentes mas com exatamente as mesmas características registradas.

Vamos verificar quais valores estão nas variáveis categóricas:

In [ ]:
categorical_columns = [
    
    "fuel",
    "seller_type",
    "transmission",
    "owner",
]

for column in categorical_columns:
    print(f"\n {column}: ")
    print(data[column].value_counts(dropna=False))
    #print(data[column].astype(str).str.strip().unique) verificação de espaços

A inspeção das variáveis categóricas não revelou inconsistências aparentes de nomenclatura, como diferenças de capitalização, erros de escrita ou categorias duplicadas semanticamente. Dessa forma, não foi identificado neste momento um tratamento de padronização necessário para essas variáveis.